# Assignment 10 - Problem 1
## Turning an English request into SQL with the OpenAI API

**Name:** Hussein Jawad

The request I need to convert to SQL:

> "Select all customers who are born in either June, July, October, live in Boston, are senior citizens and hence qualify for Medicare benefit"

What I do in this notebook:
1. Set up the OpenAI client and API key (part a).
2. Ask the model for the SQL, first with the simple prompt from the lab and then with a better prompt that returns JSON (parts b and d).
3. Check if the SQL is actually correct by running it on a small test database.
4. Try different temperatures, top_p, models and reasoning effort and compare the accuracy (part c).
5. Save and show the final JSON response (part d).

## Step 0 - OpenAI account and API key

I created an account on platform.openai.com, added billing credits, and created a new secret key under **API keys**. I named the key `e89-hw10-key`. The key is stored in a `.env` file in the same folder as this notebook, not in the notebook itself:

```
OPENAI_API_KEY=sk-...
OPENAI_KEY_NAME=e89-hw10-key
```

<!-- Paste the screenshot of the API keys page here (in Colab you can drag the image into this cell) -->

## Step 1 - Install the OpenAI library and set up the client (part a)

In [ ]:
%pip install -q --upgrade openai python-dotenv   # pandas is already in Colab, so I don't upgrade it

I run the notebook in Google Colab, and the notebook and `.env` file are in my Google Drive folder. Colab starts in `/content`, so I mount Drive and change into that folder first. When the notebook runs locally, this cell does nothing.

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/CSCI-E-89-1-Deep-Learning/HW10"   # folder in Google Drive with this notebook and .env

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir(PROJECT_DIR)
except ImportError:
    pass   # not running in Colab

print("Working folder:", os.getcwd())
print(".env found    :", os.path.exists(".env"))

Load the key from `.env` and create the client. If OpenAI returns `model_not_found`, change `MODEL` / `SMALL_MODEL`.

In [ ]:
import json
import openai
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True), override=True)

MODEL = "gpt-5.4"
SMALL_MODEL = "gpt-5.4-mini"
client = OpenAI()   # reads OPENAI_API_KEY from the environment

key = os.getenv("OPENAI_API_KEY")
assert key, "OPENAI_API_KEY not found - check the .env file"
print("openai version:", openai.__version__)
print("Key name      :", os.getenv("OPENAI_KEY_NAME"))
print("Key           :", key[:3] + "..." + key[-4:])

def show_json(obj):
    display(json.loads(obj.model_dump_json()))

## Step 2 - Table definitions and a first try (part b)

The assignment doesn't give a table, so I made a `Customers` table. It needs a `DateOfBirth` column to get the birth month and the age. I also added an `Orders` table that the query should not need. Being a senior citizen who qualifies for Medicare means being 65 or older.

In [ ]:
SCHEMA = '''
CREATE TABLE Customers (
  CustomerID   INTEGER PRIMARY KEY,
  FirstName    VARCHAR(50),
  LastName     VARCHAR(50),
  DateOfBirth  DATE,          -- 'YYYY-MM-DD'
  Email        VARCHAR(100),
  Phone        VARCHAR(20),
  City         VARCHAR(50),
  State        CHAR(2)        -- e.g. 'MA'
);

CREATE TABLE Orders (
  OrderID      INTEGER PRIMARY KEY,
  CustomerID   INTEGER REFERENCES Customers(CustomerID),
  OrderDate    DATE,
  TotalAmount  DECIMAL(10, 2)
);
'''.strip()

REQUEST = ("Select all customers who are born in either June, July, October, live in Boston, "
           "are senior citizens and hence qualify for Medicare benefit")

First I used the same kind of prompt as the SQL example in the lab: the tables go in the system message and the request in the user message.

In [ ]:
baseline_messages = [
    {"role": "system",
     "content": "Given the following SQL tables, your job is to write queries given a user's request.\n\n" + SCHEMA},
    {"role": "user", "content": REQUEST},
]

baseline_response = client.responses.create(
    model=MODEL,
    input=baseline_messages,
    reasoning={"effort": "none"},   # needed to be able to set temperature
    temperature=0,
    max_output_tokens=1000,
)
print(baseline_response.output_text)

This answer is plain text, usually with the SQL inside a code block, and since the prompt doesn't say which SQL dialect to use, the model picks one itself. That makes it hard to use the output in code, so I changed the prompt in Step 3.

## Step 3 - Better prompt with a JSON answer (parts b, c, d)

What I changed compared to the first prompt:
- The request goes between `<request>` tags and the tables between `<schema>` tags (delimiters from Notebook 1).
- I ask for SQLite, so I can run the query inside this notebook, and tell it not to hard-code today's date.
- It has to list the conditions it found before it writes the SQL, so it works the problem out first.
- The answer has to be JSON. Instead of only asking for JSON in the prompt, I used Structured Outputs (`text.format` with a JSON schema), so the reply always parses.

In [ ]:
SYSTEM_PROMPT = f'''
You are an expert SQL developer. Translate the request delimited by <request></request>
into one SQL query for this database:

<schema>
{SCHEMA}
</schema>

Rules:
- Use SQLite syntax and only the tables and columns above.
- Write a single read-only SELECT statement.
- Dates are text 'YYYY-MM-DD'. Use date('now') for today, never a hard-coded date.
- If a condition depends on age, compute the exact age.
- "Boston" means City = 'Boston' and State = 'MA'.
- "Select all customers" means return all columns.
- First list the conditions you found in the request, then write the query.
- If the request can't be answered with these tables, set "sql" to null and explain why.
'''.strip()

SQL_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "conditions":  {"type": "array", "items": {"type": "string"}},
        "assumptions": {"type": "array", "items": {"type": "string"}},
        "sql":         {"type": ["string", "null"]},
        "explanation": {"type": "string"},
    },
    "required": ["conditions", "assumptions", "sql", "explanation"],
    "additionalProperties": False,
}
TEXT_FORMAT = {"format": {"type": "json_schema", "name": "nl_to_sql",
                          "schema": SQL_JSON_SCHEMA, "strict": True}}

I use one function for all calls so I can change the prompt, the model and the settings easily. Following Notebook 1, temperature and top_p are only allowed when reasoning effort is `"none"`, so the function leaves them out for other effort levels.

In [ ]:
def generate_sql(prompt="engineered", model=MODEL, temperature=None, top_p=None, effort="none"):
    if prompt == "baseline":
        args = {"input": baseline_messages}
    else:
        args = {"input": [{"role": "system", "content": SYSTEM_PROMPT},
                          {"role": "user", "content": f"<request>{REQUEST}</request>"}],
                "text": TEXT_FORMAT}
    if effort == "none":
        if temperature is not None: args["temperature"] = temperature
        if top_p is not None:       args["top_p"] = top_p
    max_tokens = 1000 if effort == "none" else 4000   # reasoning tokens count too
    return client.responses.create(model=model, reasoning={"effort": effort},
                                   max_output_tokens=max_tokens, **args)

response = generate_sql("engineered", MODEL, temperature=0)
result = json.loads(response.output_text)
print("status:", response.status)
print(json.dumps(result, indent=2))
print("\nSQL:\n" + result["sql"])

## Step 4 - Checking if the SQL is correct

A query can look right and still be wrong, so I tested it. I made a small SQLite database where each customer tests one condition (wrong month, wrong city, too young, exactly 65, missing birth date). I worked out the correct answer in Python and compared it with what the query returns. The birth dates are based on today's date, so the test still works if I run it later.

In [ ]:
import re, sqlite3
import pandas as pd
from datetime import date, datetime, timezone

TODAY = datetime.now(timezone.utc).date()   # SQLite's date('now') uses UTC

def born(years_ago, month, day):
    return date(TODAY.year - years_ago, month, day).isoformat()

# (id, first, last, date of birth, city, state)   what the row tests
CUSTOMERS = [
    (1,  "Mary",  "Smith",  born(72, 6, 15),  "Boston",    "MA"),   # match (June)
    (2,  "John",  "Brown",  born(80, 7, 4),   "Boston",    "MA"),   # match (July)
    (3,  "Ann",   "Lee",    born(66, 10, 31), "Boston",    "MA"),   # match (October)
    (4,  "Eve",   "Clark",  born(65, 10, 1),  "Boston",    "MA"),   # exactly 65 (>= vs >)
    (5,  "Lena",  "Wong",   born(85, 6, 30),  "Boston",    "MA"),   # match
    (6,  "Bob",   "Jones",  born(75, 5, 10),  "Boston",    "MA"),   # wrong month
    (7,  "Grace", "Hill",   born(90, 12, 1),  "Boston",    "MA"),   # wrong month
    (8,  "Carl",  "Davis",  born(70, 6, 1),   "Cambridge", "MA"),   # wrong city
    (9,  "Dana",  "White",  born(40, 6, 20),  "Boston",    "MA"),   # too young
    (10, "Frank", "Green",  born(64, 7, 15),  "Boston",    "MA"),   # 63 or 64, too young
    (11, "Omar",  "King",   None,             "Boston",    "MA"),   # no birth date
]

def build_test_db():
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    con.executemany("INSERT INTO Customers VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                    [(i, f, l, dob, f.lower() + "@example.com", "617-555-01%02d" % i, city, st)
                     for i, f, l, dob, city, st in CUSTOMERS])
    con.execute("INSERT INTO Orders VALUES (1, 1, '2026-01-05', 120.50)")
    con.execute("PRAGMA query_only = ON")   # the generated SQL can't change anything
    return con

def age(dob):
    d = date.fromisoformat(dob)
    return TODAY.year - d.year - ((TODAY.month, TODAY.day) < (d.month, d.day))

# correct answer, computed in Python
EXPECTED = {i for i, f, l, dob, city, st in CUSTOMERS
            if dob and date.fromisoformat(dob).month in (6, 7, 10)
            and city == "Boston" and st == "MA" and age(dob) >= 65}
print("Today:", TODAY, "- expected CustomerIDs:", sorted(EXPECTED))

`check_sql` runs a query on the test database and compares the returned IDs with the expected ones. For the first prompt it takes the SQL out of the code block. It refuses anything that isn't a SELECT. To make sure the check itself works, I tested it with my own query and with a version that has an age mistake.

In [ ]:
def extract_sql(text):
    m = re.search(r"```(?:sql|sqlite)?\s*(.*?)```", text, re.S | re.I)
    return (m.group(1) if m else text).strip()

def check_sql(sql):
    out = {"runs": False, "correct": False, "ids": None, "error": None}
    if not sql or not re.match(r"^\s*(WITH|SELECT)\b", sql, re.I):
        out["error"] = "no SELECT statement"
        return out
    try:
        cur = build_test_db().execute(sql)
        cols = [c[0].lower() for c in cur.description]
        ids = {row[cols.index("customerid")] for row in cur.fetchall()}
        out.update(runs=True, ids=sorted(ids), correct=(ids == EXPECTED))
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {e}"
    return out

MY_SQL = '''
SELECT *
FROM Customers
WHERE CAST(strftime('%m', DateOfBirth) AS INTEGER) IN (6, 7, 10)
  AND City = 'Boston' AND State = 'MA'
  AND DateOfBirth <= date('now', '-65 years');
'''
print("my query        :", check_sql(MY_SQL))
print("age > 65 mistake:", check_sql(MY_SQL.replace("<=", "<").replace("-65", "-66")))

In [ ]:
baseline_sql = extract_sql(baseline_response.output_text)
print("First prompt :", check_sql(baseline_sql))
print("Better prompt:", check_sql(result["sql"]))

pd.read_sql_query(result["sql"], build_test_db())

## Step 5 - Trying different settings (part c)

From Notebook 1: temperature controls how random the output is (0 = almost always the same), top_p limits the choice to the most likely tokens, and reasoning effort lets the model think first (temperature and top_p can't be used then). I also compared the smaller model. I ran every setting 3 times, checked each query with `check_sql`, and counted how many different queries each setting gave.

In [ ]:
import time

SETTINGS = [
    ("first prompt, T=0",      "baseline",   MODEL,       {"temperature": 0}),
    ("better prompt, T=0",     "engineered", MODEL,       {"temperature": 0}),
    ("better prompt, T=0.7",   "engineered", MODEL,       {"temperature": 0.7}),
    ("better prompt, T=1.5",   "engineered", MODEL,       {"temperature": 1.5}),
    ("better prompt, top_p=0.1", "engineered", MODEL,     {"top_p": 0.1}),
    ("better prompt, mini, T=0", "engineered", SMALL_MODEL, {"temperature": 0}),
    ("better prompt, effort=low", "engineered", MODEL,    {"effort": "low"}),
]
N_RUNS = 3

rows = []
for label, prompt, model, settings in SETTINGS:
    for run in range(N_RUNS):
        start = time.time()
        try:
            r = generate_sql(prompt, model, **settings)
            sql = extract_sql(r.output_text) if prompt == "baseline" else json.loads(r.output_text)["sql"]
            rows.append({"setting": label, "sql": sql, "tokens": r.usage.total_tokens,
                         "seconds": time.time() - start, **check_sql(sql)})
        except Exception as e:   # e.g. a setting the model doesn't support
            rows.append({"setting": label, "sql": None, "runs": False, "correct": False,
                         "error": f"{type(e).__name__}: {e}"[:150]})
    print("done:", label)

runs = pd.DataFrame(rows)
summary = runs.groupby("setting", sort=False).agg(
    accuracy=("correct", "mean"),
    runs_on_sqlite=("runs", "mean"),
    different_queries=("sql", lambda s: s.dropna().str.strip().nunique()),
    avg_tokens=("tokens", "mean"),
    avg_seconds=("seconds", "mean"),
).round(2)
summary

In [ ]:
# runs that were not correct, and why
runs.loc[~runs["correct"], ["setting", "error", "ids", "sql"]]

### Observations

<!-- Check these against the tables above and edit if my results are different -->

- The first prompt doesn't say which SQL dialect to use, so it can use functions like `MONTH()` or `CURDATE()` that don't exist in SQLite. The logic can be right, but the query fails on my database. Saying the dialect and the date format in the prompt fixed this.
- With Structured Outputs, every run of the better prompt returned valid JSON, at all temperatures.
- At temperature 0 the query was the same (or almost the same) every time. Higher temperature changed the wording more, like aliases or a different way to compute the age. Same answer every time doesn't mean correct, which is why I needed the test database.
- top_p = 0.1 behaved a lot like a low temperature.
- Reasoning effort "low" used more tokens and took longer, and this task is simple enough that it didn't need it.
- The smaller model is cheaper; I checked if it handled the exactly-65 case correctly.

I used the better prompt with `gpt-5.4`, temperature 0 and no reasoning for the final answer: it was accurate, consistent and cheap. I would still never run generated SQL on a real database without checking it first.

## Step 6 - Final JSON response (part d)

The full API response as JSON. The model's answer is the text in `output[0].content[0].text`:

In [ ]:
final_response = generate_sql("engineered", MODEL, temperature=0)
show_json(final_response)

In [ ]:
final_answer = json.loads(final_response.output_text)

final = {
    "name": "Hussein Jawad",
    "request": REQUEST,
    "model": final_response.model,
    "settings": {"temperature": 0, "reasoning_effort": "none", "output": "json_schema"},
    "tokens": final_response.usage.total_tokens,
    "answer": final_answer,
    "test_result": check_sql(final_answer["sql"]),
}
print(json.dumps(final, indent=2))

os.makedirs("results", exist_ok=True)
with open("results/problem1_final_response.json", "w") as f:
    json.dump(final, f, indent=2)
with open("results/problem1_full_api_response.json", "w") as f:
    f.write(final_response.model_dump_json(indent=2))
runs.to_csv("results/problem1_experiments.csv", index=False)
print("\nSaved to the results folder.")